In [0]:
import pyspark.sql.functions as F

dbutils.widgets.text("TEMP_WATER_FROST", "-1")
dbutils.widgets.text("PRECIP_DRY_MAX", "0")
dbutils.widgets.text("PRECIP_LIGHT_MAX", "2")

TEMP_WATER_FROST = int(dbutils.widgets.get("TEMP_WATER_FROST"))
PRECIP_DRY_MAX = int(dbutils.widgets.get("PRECIP_DRY_MAX"))
PRECIP_LIGHT_MAX = int(dbutils.widgets.get("PRECIP_LIGHT_MAX"))


dbutils.widgets.text("silver_catalog", "dbr_dev")
dbutils.widgets.text("silver_schema", "artemzharkov10_silver")
dbutils.widgets.text("gold_catalog", "dbr_dev")
dbutils.widgets.text("gold_schema", "artemzharkov10_gold")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
GOLD_CATALOG = dbutils.widgets.get("gold_catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

SILVER_TABLE = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_history_weather_demo"
GOLD_TABLE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_history_weather_demo"
GOLD_WEIGHTS_TABLE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_weather_cluster_weights"

df_silver = spark.read.table(SILVER_TABLE)
df_weights = spark.read.table(GOLD_WEIGHTS_TABLE)

# Формирование кластеров погоды
df_clustered = (
    df_silver
    .withColumn(
        "temp_claster",
        F.when(F.col("soil_temperature_c") <= TEMP_WATER_FROST, "Frost")
         .when(F.col("soil_temperature_c") > TEMP_WATER_FROST, "Warm")
    )
    .withColumn(
        "precipitation_claster",
        F.when(F.col("precipitation_mm") == PRECIP_DRY_MAX, "Dry")
         .when((F.col("precipitation_mm") > PRECIP_DRY_MAX) & (F.col("precipitation_mm") <= PRECIP_LIGHT_MAX), "LightRain")
         .when(F.col("precipitation_mm") > PRECIP_LIGHT_MAX, "HeavyRain")
    )
    .withColumn(
        "weather_claster",
        F.concat_ws("_", F.col("temp_claster"), F.col("precipitation_claster"))
    )
)

# Интеграция (Join) с таблицей весов
df_final = (
    df_clustered.join(
        F.broadcast(df_weights),
        on=["weather_claster"],
        how="left"
    )
    .select(
        F.col("grid_id").alias("ID"),
        F.col("longitude"),
        F.col("latitude"),
        F.col("soil_temperature_c"),
        F.col("precipitation_mm"),
        F.col("weather_claster"),
        F.col("normalized_risk_index").alias("hazard_risk"),
        F.col("weather_time")
    )
)

df_final.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)
print(f"Gold таблица {GOLD_TABLE} создана. Записей: {df_final.count()}")

In [0]:
display(df_final.filter(F.col("precipitation_mm") > 1))